# AES desde cero — Notebook de estudio (Lab 3)

Implementación de **AES-128 / 192 / 256** para entender cada parte antes de pasarla a VS Code.
Prioridad aquí: **claridad y funciones legibles**, no la versión más rápida (eso se optimiza después con tablas).

Estructura del notebook, de abajo hacia arriba por dependencias:

1. `GF(2⁸)` — aritmética de campo (`xtime`, `mul`)
2. `S-box` — tabla de sustitución y su inversa
3. Representación del **state** y utilidades
4. Las cuatro transformaciones y sus inversas
5. **Key schedule** (expansión de llave) para 128/192/256
6. **Cipher**: encrypt / decrypt de un bloque
7. Validación con **test vectors de FIPS-197**

Cada sección termina con una verificación. Si todas imprimen `OK`, la implementación es correcta.


## 1. GF(2⁸) — aritmética de campo

Todo AES opera sobre bytes tratados como polinomios en el campo GF(2⁸).
- **Suma = XOR** (los coeficientes viven en {0,1}, sin acarreo).
- **Multiplicar por 2 = `xtime`**: shift a la izquierda y, si el bit 7 estaba encendido,
  XOR con `0x1B` (reducción módulo el polinomio irreducible `x⁸+x⁴+x³+x+1`).
- **Multiplicación general `mul(a,b)`**: descompone `b` en potencias de 2 leyendo sus bits;
  `a` se va duplicando con `xtime` y se suma (XOR) cuando el bit correspondiente está encendido.


In [ ]:
def xtime(b):
    """Multiplica un byte por 0x02 en GF(2^8).

    Paso 1: recuerda si el bit 7 estaba encendido ANTES de mover (b & 0x80).
    Paso 2: corre a la izquierda (multiplicar por x).
    Paso 3: si el bit 7 estaba encendido, el resultado se desbordó a x^8,
            así que se reduce con XOR 0x1B. Se recorta a 8 bits con & 0xFF.
    """
    result = b << 1
    if b & 0x80:
        result ^= 0x1B
    return result & 0xFF


def mul(a, b):
    """Multiplica a * b en GF(2^8).

    Recorre los bits de b (de menor a mayor). En cada vuelta:
      - si el bit bajo de b esta encendido, suma (XOR) la 'a' actual al resultado,
      - duplica 'a' con xtime (para representar la siguiente potencia de 2),
      - corre 'b' a la derecha para leer el siguiente bit.
    """
    result = 0
    while b:
        if b & 1:
            result ^= a
        a = xtime(a)
        b >>= 1
    return result

In [ ]:
# --- Verificacion GF ---
assert xtime(0x1d) == 0x3a          # caso sin reduccion
assert xtime(0x87) == 0x15          # caso con reduccion (bit 7 encendido)
assert mul(0x57, 0x13) == 0xfe      # test vector clasico de GF
assert mul(0x57, 0x01) == 0x57      # por 1 no cambia
assert mul(0x57, 0x02) == xtime(0x57)  # por 2 == xtime
print("GF OK")

## 2. S-box

La S-box de AES se define como: **inverso multiplicativo en GF(2⁸)** seguido de una
**transformación afín** (una mezcla de bits fija + XOR con `0x63`).

Aquí la **generamos** desde esa definición (en vez de copiar la tabla), lo que:
- verifica de paso que tu `mul`/`xtime` funcionan,
- es más fácil de defender ("no la copié, la construí").

Pasos:
1. Calcular el inverso multiplicativo de cada byte en GF(2⁸).
   El de `0x00` se define como `0x00` (no tiene inverso, es un caso especial del estándar).
2. Aplicar la transformación afín a cada inverso.

La **inversa** (`INV_SBOX`) se obtiene simplemente invirtiendo la tabla:
si `SBOX[a] = b`, entonces `INV_SBOX[b] = a`.


In [ ]:
def _gf_inverse(a):
    """Inverso multiplicativo de 'a' en GF(2^8) por busqueda exhaustiva.

    Solo se usa UNA vez para construir la S-box, asi que la fuerza bruta
    (probar los 256 bytes) es perfectamente aceptable. inverso de 0 = 0.
    """
    if a == 0:
        return 0
    for b in range(256):
        if mul(a, b) == 1:
            return b
    return 0


def _affine(b):
    """Transformacion afin de la S-box aplicada a un byte ya invertido.

    Cada bit de salida es un XOR de 5 bits del byte rotado, mas el bit
    correspondiente de la constante 0x63. La forma compacta usa rotaciones:
      s = b ^ rotl(b,1) ^ rotl(b,2) ^ rotl(b,3) ^ rotl(b,4) ^ 0x63
    donde rotl es rotacion circular de 8 bits a la izquierda.
    """
    def rotl(x, n):
        return ((x << n) | (x >> (8 - n))) & 0xFF
    return b ^ rotl(b, 1) ^ rotl(b, 2) ^ rotl(b, 3) ^ rotl(b, 4) ^ 0x63


def _build_sbox():
    """Construye SBOX e INV_SBOX desde la definicion matematica."""
    sbox = [0] * 256
    for a in range(256):
        sbox[a] = _affine(_gf_inverse(a))
    inv = [0] * 256
    for a in range(256):
        inv[sbox[a]] = a          # si sbox[a]=b entonces inv[b]=a
    return sbox, inv


SBOX, INV_SBOX = _build_sbox()

In [ ]:
# --- Verificacion S-box ---
# Valores conocidos del estandar FIPS-197
assert SBOX[0x00] == 0x63
assert SBOX[0x53] == 0xed
assert SBOX[0xc2] == 0x25
assert SBOX[0x9a] == 0xb8
# La inversa debe deshacer la S-box para todos los bytes
assert all(INV_SBOX[SBOX[x]] == x for x in range(256))
print("S-box OK")

## 3. Representación del state y utilidades

El **state** son los 16 bytes del bloque. Se guardan en un `bytearray` **plano** de 16 posiciones,
ordenados **por columnas** (convención de AES):

```
indice en el bytearray:      posicion (fila, columna):
 0   4   8  12                (0,0) (0,1) (0,2) (0,3)
 1   5   9  13                (1,0) (1,1) (1,2) (1,3)
 2   6  10  14                (2,0) (2,1) (2,2) (2,3)
 3   7  11  15                (3,0) (3,1) (3,2) (3,3)
```

El byte en (fila `r`, columna `c`) está en el índice `r + 4*c`.
Los primeros 4 bytes de entrada forman la **primera columna**, no la primera fila.


In [ ]:
def bytes_to_state(data16):
    """Convierte 16 bytes en un state (bytearray plano por columnas).

    Como los datos ya vienen en el orden r+4*c que AES usa, esto es una
    copia directa a bytearray. Se hace explicito para dejar clara la conancion.
    """
    assert len(data16) == 16
    return bytearray(data16)


def state_to_bytes(state):
    """Convierte el state de vuelta a bytes (mismo orden por columnas)."""
    return bytes(state)


def print_state(state):
    """Imprime el state como matriz 4x4 en hex, para depurar.

    Recorre por filas (r) y dentro de cada fila por columnas (c),
    leyendo el indice r+4*c.
    """
    for r in range(4):
        print(" ".join(f"{state[r + 4*c]:02x}" for c in range(4)))

In [ ]:
# --- Verificacion state ---
_demo = bytes.fromhex("32 43 f6 a8 88 5a 30 8d 31 31 98 a2 e0 37 07 34")
_st = bytes_to_state(_demo)
# primera columna = primeros 4 bytes
assert _st[0] == 0x32 and _st[1] == 0x43 and _st[2] == 0xf6 and _st[3] == 0xa8
# el byte (fila 0, columna 1) es el quinto byte
assert _st[0 + 4*1] == 0x88
assert state_to_bytes(_st) == _demo
print("state OK")
print_state(_st)

## 4. Las cuatro transformaciones (y sus inversas)

Cada round aplica, en orden: **SubBytes → ShiftRows → MixColumns → AddRoundKey**.
El descifrado usa las inversas. Todas operan sobre el state de 16 bytes.


### 4.1 SubBytes / InvSubBytes

Sustituye **cada byte** del state por su valor en la S-box (o en la inversa).
Es la única capa no lineal (confusión). Byte a byte, independiente.


In [ ]:
def sub_bytes(state):
    """Reemplaza cada byte del state por SBOX[byte]."""
    for i in range(16):
        state[i] = SBOX[state[i]]


def inv_sub_bytes(state):
    """Reemplaza cada byte del state por INV_SBOX[byte]."""
    for i in range(16):
        state[i] = INV_SBOX[state[i]]

### 4.2 ShiftRows / InvShiftRows

Rota cada **fila** del state. Fila `r` rota `r` posiciones (fila 0 no se mueve).
- `ShiftRows`: rota a la **izquierda**.
- `InvShiftRows`: rota a la **derecha** (deshace).

La fila `r` ocupa los índices `r, r+4, r+8, r+12`.


In [ ]:
def shift_rows(state):
    """Rota cada fila r a la izquierda r posiciones."""
    for r in range(1, 4):                      # fila 0 no se mueve
        row = [state[r + 4*c] for c in range(4)]   # leer la fila
        row = row[r:] + row[:r]                     # rotar r a la izquierda
        for c in range(4):
            state[r + 4*c] = row[c]                 # escribir de vuelta


def inv_shift_rows(state):
    """Rota cada fila r a la derecha r posiciones (inversa de shift_rows)."""
    for r in range(1, 4):
        row = [state[r + 4*c] for c in range(4)]
        row = row[-r:] + row[:-r]                   # rotar r a la derecha
        for c in range(4):
            state[r + 4*c] = row[c]

### 4.3 MixColumns / InvMixColumns

Multiplica cada **columna** (4 bytes) por una matriz fija en GF(2⁸).

MixColumns (cifrado), constantes 02/03/01:
```
02 03 01 01
01 02 03 01
01 01 02 03
03 01 01 02
```

InvMixColumns (descifrado), constantes 0e/0b/0d/09 (más caras → descifrado más lento):
```
0e 0b 0d 09
09 0e 0b 0d
0d 09 0e 0b
0b 0d 09 0e
```

Cada columna `c` ocupa los índices `4c, 4c+1, 4c+2, 4c+3`.


In [ ]:
def mix_columns(state):
    """Multiplica cada columna por la matriz fija de MixColumns en GF(2^8)."""
    for c in range(4):
        i = 4 * c
        s0, s1, s2, s3 = state[i], state[i+1], state[i+2], state[i+3]
        state[i]   = mul(s0, 2) ^ mul(s1, 3) ^ s2 ^ s3          # 02 03 01 01
        state[i+1] = s0 ^ mul(s1, 2) ^ mul(s2, 3) ^ s3          # 01 02 03 01
        state[i+2] = s0 ^ s1 ^ mul(s2, 2) ^ mul(s3, 3)          # 01 01 02 03
        state[i+3] = mul(s0, 3) ^ s1 ^ s2 ^ mul(s3, 2)          # 03 01 01 02


def inv_mix_columns(state):
    """Multiplica cada columna por la matriz inversa de MixColumns."""
    for c in range(4):
        i = 4 * c
        s0, s1, s2, s3 = state[i], state[i+1], state[i+2], state[i+3]
        state[i]   = mul(s0, 0x0e) ^ mul(s1, 0x0b) ^ mul(s2, 0x0d) ^ mul(s3, 0x09)
        state[i+1] = mul(s0, 0x09) ^ mul(s1, 0x0e) ^ mul(s2, 0x0b) ^ mul(s3, 0x0d)
        state[i+2] = mul(s0, 0x0d) ^ mul(s1, 0x09) ^ mul(s2, 0x0e) ^ mul(s3, 0x0b)
        state[i+3] = mul(s0, 0x0b) ^ mul(s1, 0x0d) ^ mul(s2, 0x09) ^ mul(s3, 0x0e)

### 4.4 AddRoundKey

XOR del state con los 16 bytes de la round key. Es su propia inversa (por eso no hay
`inv_add_round_key`). `round_key` es un bytearray/bytes de 16 posiciones.


In [ ]:
def add_round_key(state, round_key):
    """XOR posicion a posicion del state con la round key (16 bytes)."""
    for i in range(16):
        state[i] ^= round_key[i]

In [ ]:
# --- Verificacion transformaciones (usando el ejemplo de FIPS-197) ---
# Estado despues de SubBytes en el round 1 del ejemplo del apendice B.
_s = bytes_to_state(bytes.fromhex("d4 27 11 ae e0 bf 98 f1 b8 b4 5d e5 1e 41 52 30"))
shift_rows(_s)
assert state_to_bytes(_s) == bytes.fromhex("d4bf5d30e0b452aeb84111f11e2798e5")

# ShiftRows seguido de InvShiftRows debe devolver el original
_s2 = bytes_to_state(bytes.fromhex("00112233445566778899aabbccddeeff"))
_orig = bytes(_s2)
shift_rows(_s2); inv_shift_rows(_s2)
assert bytes(_s2) == _orig

# MixColumns con una columna conocida (NO constante) del ejemplo FIPS-197
_s3 = bytes_to_state(bytes.fromhex("d4 bf 5d 30 e0 b4 52 ae b8 41 11 f1 1e 27 98 e5"))
mix_columns(_s3)
assert state_to_bytes(_s3) == bytes.fromhex("046681e5e0cb199a48f8d37a2806264c")

# MixColumns seguido de InvMixColumns debe devolver el original
_s4 = bytes_to_state(bytes.fromhex("00112233445566778899aabbccddeeff"))
_orig4 = bytes(_s4)
mix_columns(_s4); inv_mix_columns(_s4)
assert bytes(_s4) == _orig4
print("transformaciones OK")

## 5. Key schedule (expansión de llave)

Expande la llave original en una lista de **round keys** de 16 bytes cada una.
Se trabaja en **palabras** de 4 bytes.

Parámetros según el tamaño de llave:

| Versión | Nk (palabras de llave) | Nr (rounds) | round keys |
|--------|----|----|----|
| AES-128 | 4 | 10 | 11 |
| AES-192 | 6 | 12 | 13 |
| AES-256 | 8 | 14 | 15 |

Se generan `4*(Nr+1)` palabras. La palabra `i`:
- normalmente: `W[i] = W[i-Nk] ⊕ W[i-1]`
- si `i % Nk == 0`: `W[i] = W[i-Nk] ⊕ g(W[i-1])`, con `g = SubWord(RotWord(w)) ⊕ Rcon`
- **solo en AES-256** (Nk=8), si `i % Nk == 4`: `W[i] = W[i-Nk] ⊕ SubWord(W[i-1])`
  (este caso extra es el bug más común de AES-256)


In [ ]:
# Constantes de round (Rcon). Cada una es [RC, 0, 0, 0]; aqui guardamos solo RC.
# RC[i] = xtime aplicado sucesivamente: 01,02,04,08,10,20,40,80,1b,36,...
RCON = [0x01, 0x02, 0x04, 0x08, 0x10, 0x20, 0x40, 0x80, 0x1b, 0x36,
        0x6c, 0xd8, 0xab, 0x4d]   # suficientes para las 3 versiones


def _rot_word(w):
    """Rota una palabra de 4 bytes una posicion a la izquierda: [a,b,c,d]->[b,c,d,a]."""
    return w[1:] + w[:1]


def _sub_word(w):
    """Aplica la S-box a cada uno de los 4 bytes de la palabra."""
    return [SBOX[b] for b in w]


def key_expansion(key):
    """Expande la llave (16, 24 o 32 bytes) en round keys de 16 bytes.

    Devuelve (round_keys, Nr) donde round_keys es una lista de bytearrays de 16.
    """
    Nk = len(key) // 4                 # 4, 6 u 8 palabras
    Nr = {4: 10, 6: 12, 8: 14}[Nk]     # rounds segun el tamano

    # W: lista de palabras (cada una lista de 4 bytes). Las primeras Nk vienen de la llave.
    W = [list(key[4*i:4*i+4]) for i in range(Nk)]

    for i in range(Nk, 4 * (Nr + 1)):
        temp = list(W[i-1])
        if i % Nk == 0:
            temp = _sub_word(_rot_word(temp))
            temp[0] ^= RCON[i // Nk - 1]        # XOR Rcon en el primer byte
        elif Nk == 8 and i % Nk == 4:
            temp = _sub_word(temp)              # caso extra SOLO en AES-256
        W.append([W[i-Nk][j] ^ temp[j] for j in range(4)])

    # Agrupar cada 4 palabras en una round key de 16 bytes
    round_keys = []
    for r in range(Nr + 1):
        rk = bytearray()
        for w in range(4):
            rk.extend(W[4*r + w])
        round_keys.append(rk)
    return round_keys, Nr

In [ ]:
# --- Verificacion key schedule (AES-128, ejemplo FIPS-197 apendice A) ---
_rks, _nr = key_expansion(bytes.fromhex("2b7e151628aed2a6abf7158809cf4f3c"))
assert _nr == 10 and len(_rks) == 11
assert bytes(_rks[0]) == bytes.fromhex("2b7e151628aed2a6abf7158809cf4f3c")   # round key 0 = llave
assert bytes(_rks[1]) == bytes.fromhex("a0fafe1788542cb123a339392a6c7605")   # round key 1
assert bytes(_rks[10]) == bytes.fromhex("d014f9a8c9ee2589e13f0cc8b6630ca6")  # round key 10

# Numero de rounds correcto para cada version
assert key_expansion(b"\x00"*16)[1] == 10
assert key_expansion(b"\x00"*24)[1] == 12
assert key_expansion(b"\x00"*32)[1] == 14
print("key schedule OK")

## 6. Cipher — encrypt / decrypt de un bloque

Ensambla todo. **Cifrado**:
1. `AddRoundKey` inicial (round key 0)
2. `Nr-1` rounds completos: SubBytes → ShiftRows → MixColumns → AddRoundKey
3. round final (sin MixColumns): SubBytes → ShiftRows → AddRoundKey

**Descifrado**: las inversas en orden espejo, con las round keys al revés.


In [ ]:
def encrypt_block(plaintext16, key):
    """Cifra un bloque de 16 bytes con la llave dada (16/24/32 bytes)."""
    round_keys, Nr = key_expansion(key)
    state = bytes_to_state(plaintext16)

    add_round_key(state, round_keys[0])              # AddRoundKey inicial

    for r in range(1, Nr):                           # rounds completos
        sub_bytes(state)
        shift_rows(state)
        mix_columns(state)
        add_round_key(state, round_keys[r])

    sub_bytes(state)                                 # round final (sin MixColumns)
    shift_rows(state)
    add_round_key(state, round_keys[Nr])

    return state_to_bytes(state)


def decrypt_block(ciphertext16, key):
    """Descifra un bloque de 16 bytes (inversa exacta de encrypt_block)."""
    round_keys, Nr = key_expansion(key)
    state = bytes_to_state(ciphertext16)

    add_round_key(state, round_keys[Nr])             # deshacer round final
    inv_shift_rows(state)
    inv_sub_bytes(state)

    for r in range(Nr - 1, 0, -1):                   # rounds completos al reves
        add_round_key(state, round_keys[r])
        inv_mix_columns(state)
        inv_shift_rows(state)
        inv_sub_bytes(state)

    add_round_key(state, round_keys[0])              # deshacer AddRoundKey inicial

    return state_to_bytes(state)

## 7. Validación con test vectors de FIPS-197

Los vectores oficiales del apéndice C. Si todo imprime `OK`, la implementación
de las tres versiones es correcta.


In [ ]:
# Vector unico de plaintext usado por las 3 versiones en FIPS-197 apendice C
PT = bytes.fromhex("00112233445566778899aabbccddeeff")

# AES-128
K128 = bytes.fromhex("000102030405060708090a0b0c0d0e0f")
C128 = bytes.fromhex("69c4e0d86a7b0430d8cdb78070b4c55a")
assert encrypt_block(PT, K128) == C128
assert decrypt_block(C128, K128) == PT

# AES-192
K192 = bytes.fromhex("000102030405060708090a0b0c0d0e0f1011121314151617")
C192 = bytes.fromhex("dda97ca4864cdfe06eaf70a0ec0d7191")
assert encrypt_block(PT, K192) == C192
assert decrypt_block(C192, K192) == PT

# AES-256
K256 = bytes.fromhex("000102030405060708090a0b0c0d0e0f101112131415161718191a1b1c1d1e1f")
C256 = bytes.fromhex("8ea2b7ca516745bfeafc49904b496089")
assert encrypt_block(PT, K256) == C256
assert decrypt_block(C256, K256) == PT

print("AES-128 OK")
print("AES-192 OK")
print("AES-256 OK")

In [ ]:
# --- Prueba de ida y vuelta D(E(P)) = P con datos arbitrarios ---
import os
for key_len in (16, 24, 32):
    k = os.urandom(key_len)
    p = os.urandom(16)
    assert decrypt_block(encrypt_block(p, k), k) == p
print("round-trip D(E(P)) = P OK para 128/192/256")

## Siguientes pasos (fuera de este notebook)

Cuando lleves esto a VS Code:
- Separar en módulos: `gf.py`, `sbox.py`, `transforms.py`, `key_schedule.py`, `cipher.py`.
- Tests en `tests/` (los `assert` de aquí ya son la base).
- **Optimización para el benchmark**: precomputar tablas (la S-box ya es tabla;
  añadir tablas de multiplicación para `mul` por las constantes, y opcionalmente T-tables).
- El benchmark procesa los bloques de forma independiente (estilo ECB) sobre 1/10/100 MB;
  el propio lab aclara que evalúa el core, no un modo seguro.

📌 Recuerda verificar con el profesor si en la validación quiere manejo de mensajes de longitud
arbitraria (padding PKCS#7) además del cifrado por bloque.
